# IntegrAO

## Setup

In [11]:
%load_ext autoreload
%autoreload 2

import os
import sys

NOTEBOOK_DIR = os.getcwd()
PROJECT_ROOT = os.path.abspath(os.path.join(NOTEBOOK_DIR, '..'))
sys.path.insert(0, PROJECT_ROOT)

import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pickle
import seaborn as sns
from sklearn.cluster import spectral_clustering
from sklearn.metrics import v_measure_score
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

from utility import *
from integrao.integrater import integrao_integrater, integrao_predictor
from implementations_integrao import *

Torch version: 2.1.0+cpu
CUDA available: False
The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Torch version: 2.1.0+cpu
CUDA available: False


## Hyperparameters

In [2]:
# Hyperparameters
neighbor_size = 10
embedding_dims = 32
fusing_iteration = 30
normalization_factor = 1.0
alighment_epochs = 1000
beta = 1.0
mu = 0.5
finetune_epochs = 1000

dataset_name = 'TCGA-BRCA'
cluster_number = 4

top_var_nbr = 2000

## Data loading and preprocessing

In [3]:
data_dir = "../data/TCGA-BRCA/"

module_path = os.path.abspath(os.path.join('../data/'))
if module_path not in sys.path:
    sys.path.append(module_path)

mod = ["mRNA", "DNAm", "RPPA"]

data = {}
for omic in mod :
   with open(f"{data_dir}{omic}.pkl", "rb") as f:  # 'rb' = read binary
    data[omic] = pickle.load(f)

y_dicts = []
for omic in mod:
    y_series = data[omic]['meta']['paper_BRCA_Subtype_PAM50']
    y_dicts.append(y_series)

In [4]:
result_base_dir = os.path.dirname(os.path.dirname(data_dir))
result_dir = os.path.join(result_base_dir, "results", "TCGA-BRCA")

if not os.path.exists(result_dir):
    os.makedirs(result_dir)

In [5]:
patients_per_mod = {omic: data[omic]['expr'].index for omic in mod}

all_patients = set().union(*patients_per_mod.values())
all_patients = sorted(list(all_patients))

print(f"Total patients: {len(all_patients)}")

Total patients: 658


In [6]:
omic_matrices = {}

for omic in mod:
    X = data[omic]['expr']
    X_full = X.reindex(all_patients)
    omic_matrices[omic] = X_full

In [7]:
label_dicts = []
for omic in mod:
    y_series = data[omic]['meta']['paper_BRCA_Subtype_PAM50']
    label_dicts.append(y_series)

y = pd.Series(index=all_patients, dtype=object).to_frame(name="paper_BRCA_Subtype_PAM50")

for patient in all_patients:
    assigned = None
    for y_series in label_dicts:
        if patient in y_series.index and pd.notna(y_series.loc[patient]):
            assigned = y_series.loc[patient]
            break
    y.loc[patient] = assigned

print("Label counts:\n", y.value_counts(dropna=False))

Label counts:
 paper_BRCA_Subtype_PAM50
LumA                        352
LumB                        138
Basal                       119
Her2                         49
Name: count, dtype: int64


## Cross validation

In [8]:
# Create stratified folds
common_patients = set(omic_matrices["mRNA"].index)
for omic in ["DNAm", "RPPA"]:
    common_patients = common_patients.intersection(omic_matrices[omic].index)
common_patients = common_patients.intersection(y.index)
common_patients = sorted(list(common_patients))

print(f"Common patients across all modalities: {len(common_patients)}")

folds = create_stratified_folds(y, stratify_col="paper_BRCA_Subtype_PAM50", n_splits=5)

Common patients across all modalities: 658
Fold 1 (size=132):
paper_BRCA_Subtype_PAM50
LumA     0.530303
LumB     0.212121
Basal    0.181818
Her2     0.075758
Name: proportion, dtype: float64
----------------------------------------
Fold 2 (size=132):
paper_BRCA_Subtype_PAM50
LumA     0.530303
LumB     0.212121
Basal    0.181818
Her2     0.075758
Name: proportion, dtype: float64
----------------------------------------
Fold 3 (size=132):
paper_BRCA_Subtype_PAM50
LumA     0.537879
LumB     0.204545
Basal    0.181818
Her2     0.075758
Name: proportion, dtype: float64
----------------------------------------
Fold 4 (size=131):
paper_BRCA_Subtype_PAM50
LumA     0.541985
LumB     0.206107
Basal    0.175573
Her2     0.076336
Name: proportion, dtype: float64
----------------------------------------
Fold 5 (size=131):
paper_BRCA_Subtype_PAM50
LumA     0.534351
LumB     0.213740
Basal    0.183206
Her2     0.068702
Name: proportion, dtype: float64
----------------------------------------


In [9]:
model_path = os.path.join(result_dir, "model_integrao_supervised.pth")
fold_results, best_queries = training_integrao_crossval(folds, omic_matrices, model_path, top_var_nbr, dataset_name, neighbor_size, embedding_dims,
                            fusing_iteration, normalization_factor, alighment_epochs, beta, mu, y, result_dir, finetune_epochs, cluster_number,
                           modalities_name_list=["mRNA", "DNAm", "RPPA"])

# Extract class names from the first fold result for final summary display
label_dicts = []
for omic in mod:
    y_series = data[omic]['meta']['paper_BRCA_Subtype_PAM50']
    label_dicts.append(y_series)

y_series = y.iloc[:, 0] if isinstance(y, pd.DataFrame) else y
label_map = {label: i for i, label in enumerate(sorted([label for label in y_series.unique() if pd.notna(label)]))}
class_names = [None] * len(label_map)
for label, idx in label_map.items():
    class_names[idx] = str(label)

# Display final CV summary
display_cv_summary(fold_results, class_names=class_names)


Processing Fold 1/5
Start indexing input expression matrices!
Common sample between view0 and view1: 327
Common sample between view0 and view2: 326
Common sample between view1 and view2: 328
Neighbor size: 10
Start applying diffusion!
Diffusion ends! Times: 15.974958658218384s
Starting unsupervised exmbedding extraction!
Dataset 0: (398, 2000)
Dataset 1: (398, 2000)
Dataset 2: (395, 464)
epoch 0: loss 34.107452392578125, align_loss:0.767931
epoch 100: loss 25.95161247253418, align_loss:0.644899
epoch 200: loss 2.5975499153137207, align_loss:0.268639
epoch 300: loss 2.594761371612549, align_loss:0.266834
epoch 400: loss 2.5915164947509766, align_loss:0.265109
epoch 500: loss 2.5878469944000244, align_loss:0.263249
epoch 600: loss 2.583789825439453, align_loss:0.261148
epoch 700: loss 2.5793375968933105, align_loss:0.259035
epoch 800: loss 2.5744965076446533, align_loss:0.256709
epoch 900: loss 2.5692522525787354, align_loss:0.254336
Manifold alignment ends! Times: 90.26259183883667s
St

## Interpretation

In [12]:
# Create a fresh predictor instantiated with the datasets we want to interpret
predictor_interpret = integrao_predictor(
    best_queries,
    dataset_name,
    modalities_name_list=["mRNA", "DNAm", "RPPA"],
    neighbor_size=neighbor_size,
    embedding_dims=embedding_dims,
    fusing_iteration=fusing_iteration,
    normalization_factor=normalization_factor,
    alighment_epochs=alighment_epochs,
    beta=beta,
    mu=mu,
    num_classes=cluster_number,
)
# compute fused networks for these datasets (ensures internal indexing matches)
_ = predictor_interpret.network_diffusion()

# Run interpretation on the same datasets used for prediction (query sets)
# interpret_supervised will save per-domain CSVs and also return a list of DataFrames

df_list = predictor_interpret.interpret_supervised(
    model_path=model_path,
    result_dir=result_dir,
    new_datasets=best_queries,
    modalities_names=["mRNA", "DNAm", "RPPA"],
)

# df_list is a list of DataFrames (one per domain). Print shapes and heads for quick inspection
modalities = ["mRNA", "DNAm", "RPPA"]
for name, df in zip(modalities, df_list):
    if df is None:
        print(f"{name}: returned None")
        continue
    print(f"{name} feature importance: shape={df.shape}")
    try:
        print(df.head())
    except Exception as e:
        print(f"Could not print head() for {name}: {e}")

# Optionally keep df_list for later inspection
feat_importances = {name: df for name, df in zip(modalities, df_list)}
print('Interpretation complete, feature importances stored in `feat_importances`.')

Start indexing input expression matrices!
Common sample between view0 and view1: 86
Common sample between view0 and view2: 84
Common sample between view1 and view2: 88
Neighbor size: 10
Start applying diffusion!
Diffusion ends! Times: 4.568008899688721s
IntegrAO(
  (feature): ModuleList(
    (0-1): 2 x GraphSAGE(2000, 32, num_layers=2)
    (2): GraphSAGE(464, 32, num_layers=2)
  )
  (feature_show): Sequential(
    (0): Linear(in_features=32, out_features=32, bias=True)
    (1): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): LeakyReLU(negative_slope=0.1, inplace=True)
    (3): Linear(in_features=32, out_features=32, bias=True)
  )
  (pred_head): Sequential(
    (0): Linear(in_features=32, out_features=16, bias=True)
    (1): BatchNorm1d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): LeakyReLU(negative_slope=0.1, inplace=True)
    (3): Linear(in_features=16, out_features=4, bias=True)
  )
)
Loaded pre-trained model w

In [ ]:
# Define the number of top features to plot
num_top_features = 20

# Loop through each omic and its feature importances
for omic_name, df_importance in feat_importances.items():
    # Calculate the mean of the absolute feature importances for each feature
    mean_abs_importance = df_importance.abs().mean().sort_values(ascending=False)

    # Select the top N most important features
    top_n_features = mean_abs_importance.head(num_top_features)

    # Create the bar plot
    plt.figure(figsize=(12, 7))
    sns.barplot(x=top_n_features.index, y=top_n_features.values, palette='viridis')
    plt.title(f'Top {num_top_features} Most Important Features for {omic_name} Omic', fontsize=20)
    plt.xlabel('Feature', fontsize=20)
    plt.ylabel('Mean Absolute Importance', fontsize=20)
    plt.xticks(rotation=45, ha='right', fontsize=10)
    plt.yticks(fontsize=10)
    plt.tight_layout()
    plt.show()